In [2]:
import os
import json
import re
from datetime import datetime

import pandas as pd
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter


# =========================
# Styling helpers
# =========================
THIN = Side(style="thin", color="BFBFBF")
BORDER = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)

HEADER_FILL = PatternFill("solid", fgColor="1F4E79")
HEADER_FONT = Font(bold=True, color="FFFFFF")
TITLE_FONT = Font(bold=True, size=14, color="1F4E79")
BOLD = Font(bold=True)

CENTER = Alignment(horizontal="center", vertical="center", wrap_text=True)
LEFT = Alignment(horizontal="left", vertical="center", wrap_text=True)

def autosize_columns(ws, min_width=10, max_width=55):
    for col in ws.columns:
        max_len = 0
        col_letter = get_column_letter(col[0].column)
        for cell in col:
            v = "" if cell.value is None else str(cell.value)
            max_len = max(max_len, len(v))
        ws.column_dimensions[col_letter].width = max(min_width, min(max_width, max_len + 2))

def style_header_row(ws, header_row=1):
    for cell in ws[header_row]:
        cell.fill = HEADER_FILL
        cell.font = HEADER_FONT
        cell.alignment = CENTER
        cell.border = BORDER

def style_table(ws, start_row, start_col, end_row, end_col, header=True):
    for r in range(start_row, end_row + 1):
        for c in range(start_col, end_col + 1):
            cell = ws.cell(row=r, column=c)
            cell.border = BORDER
            if r == start_row and header:
                cell.fill = HEADER_FILL
                cell.font = HEADER_FONT
                cell.alignment = CENTER
            else:
                cell.alignment = LEFT

def write_df(ws, df, start_row=1, start_col=1, header=True):
    r = start_row
    for row in dataframe_to_rows(df, index=False, header=header):
        c = start_col
        for value in row:
            ws.cell(row=r, column=c, value=value)
            c += 1
        r += 1
    # style
    end_row = start_row + len(df) + (1 if header else 0) - 1
    end_col = start_col + df.shape[1] - 1
    style_table(ws, start_row, start_col, end_row, end_col, header=header)
    autosize_columns(ws)
    return end_row, end_col


# =========================
# Sheet name sanitization
# =========================
INVALID_SHEET_CHARS = r'[\[\]\:\*\?\/\\]'
def safe_sheet_name(name: str, suffix=""):
    """
    Excel sheet name:
      - max 31 chars
      - no []:*?/\
    """
    if name is None:
        name = "Sheet"
    name = re.sub(INVALID_SHEET_CHARS, "-", str(name)).strip()
    if suffix:
        name = f"{name}{suffix}"
    if len(name) > 31:
        name = name[:31]
    if not name:
        name = "Sheet"
    return name


# =========================
# Data extraction helpers
# =========================
def list_components(fam):
    """
    Returns a dataframe of components:
    SoleType | ComponentCode | ComponentName | Compound | Notes
    """
    rows = []
    components = fam.get("components", {})
    for sole_type, comp_dict in components.items():
        for comp_code, comp in comp_dict.items():
            rows.append({
                "SoleType": sole_type,
                "ComponentCode": comp_code,
                "ComponentName": comp.get("displayName"),
                "Compound": comp.get("compound"),
                "Notes": comp.get("notes")
            })
    return pd.DataFrame(rows)

def list_styles(fam):
    rows = []
    for s in fam.get("stylesUsingThisFamily", []) or []:
        rows.append({
            "StyleCode": s.get("styleCode"),
            "StyleName": s.get("styleName"),
            "Season": s.get("season"),
            "Notes": s.get("notes")
        })
    return pd.DataFrame(rows)

def list_sourcing_rules(fam):
    rows = []
    for r in fam.get("sourcingRules", []) or []:
        rows.append({
            "FactoryCode": r.get("factoryCode"),
            "ComponentCode": r.get("componentCode"),
            "VendorCode": r.get("vendorCode"),
            "EffectiveFrom": r.get("effectiveFrom"),
            "EffectiveTo": r.get("effectiveTo"),
            "Notes": r.get("notes")
        })
    return pd.DataFrame(rows)

def list_sizing_rules(fam):
    """
    Flattens Option A sizing rules:
    ComponentCode | SoleType | ComponentName | MoldSize | ShoeSizes | Notes

    ShoeSizes is blank if null. If filled later, it’s comma-joined string.
    """
    rows = []
    sizing = fam.get("sizingRulesByComponent", {}) or {}
    for comp_code, block in sizing.items():
        sole_type = block.get("soleType")
        comp_name = block.get("componentName")
        notes = block.get("notes")
        ms_map = block.get("moldSizeToShoeSizes", {}) or {}
        for mold_size, shoe_sizes in ms_map.items():
            if shoe_sizes is None:
                shoe_sizes_str = ""  # stakeholder will fill later
            else:
                # normalize list → "a, b, c"
                shoe_sizes_str = ", ".join([str(x) for x in shoe_sizes])
            rows.append({
                "ComponentCode": comp_code,
                "SoleType": sole_type,
                "ComponentName": comp_name,
                "MoldSize": str(mold_size),
                "ShoeSizes": shoe_sizes_str,
                "Notes": notes
            })
    df = pd.DataFrame(rows)
    # Keep a nice sort order: component then numeric-ish mold size
    if not df.empty:
        def _mold_to_float(x):
            try:
                return float(x)
            except:
                return 9999.0
        df["__m"] = df["MoldSize"].apply(_mold_to_float)
        df = df.sort_values(["ComponentCode", "__m"]).drop(columns="__m")
    return df

def build_inventory_matrix_for_component(comp):
    """
    Creates a human-friendly grid for ONE component sheet:

    Row = MoldSize
    Cols = each mold record (vendor/shop/location label)
    Values = qtyByMoldSize[size]

    Also returns a "meta" list about each column.
    """
    molds = comp.get("molds", []) or []

    # Collect all sizes used in any record
    all_sizes = set()
    for m in molds:
        qty_map = ((m.get("inventory") or {}).get("qtyByMoldSize")) or {}
        all_sizes.update([str(k) for k in qty_map.keys()])

    # Sort sizes numeric if possible
    def _to_float(s):
        try:
            return float(s)
        except:
            return 9999.0
    sizes_sorted = sorted(all_sizes, key=_to_float)

    # Build columns (one per mold record)
    col_labels = []
    col_meta = []
    for i, m in enumerate(molds, start=1):
        vendor = m.get("vendorName") or m.get("vendorCode")
        shop = m.get("moldShopName") or m.get("moldShopCode")
        loc = (m.get("location") or {}).get("name")
        # label preference: Vendor > Shop > "MoldRecord#"
        label_core = vendor or shop or f"MoldRecord{i}"
        # add location if helpful
        label = f"{label_core} ({loc})" if loc else str(label_core)
        col_labels.append(label)

        col_meta.append({
            "colLabel": label,
            "vendorCode": m.get("vendorCode"),
            "vendorName": m.get("vendorName"),
            "moldShopCode": m.get("moldShopCode"),
            "moldShopName": m.get("moldShopName"),
            "location": loc,
            "initSeason": m.get("initSeason"),
            "lastDemandSeason": m.get("lastDemandSeason"),
            "totalMoldQty": (m.get("inventory") or {}).get("totalMoldQty"),
            "dailyOutput": (m.get("capacity") or {}).get("dailyOutput"),
            "moldCost": (m.get("asset") or {}).get("moldCost"),
            "ownership": (m.get("asset") or {}).get("ownership"),
            "condition": (m.get("asset") or {}).get("condition")
        })

    # Matrix rows
    matrix_rows = []
    for s in sizes_sorted:
        row = {"MoldSize": s}
        for label, m in zip(col_labels, molds):
            qty_map = ((m.get("inventory") or {}).get("qtyByMoldSize")) or {}
            row[label] = qty_map.get(s, None)
        matrix_rows.append(row)

    df_matrix = pd.DataFrame(matrix_rows)

    return df_matrix, col_meta


# =========================
# Workbook export (per family)
# =========================
def export_one_family_to_excel(family_code, fam, output_dir):
    wb = Workbook()

    # Remove default sheet
    wb.remove(wb.active)

    # -------------------------
    # 1) Basic_information
    # -------------------------
    ws_basic = wb.create_sheet("Basic_information")
    ws_basic["A1"] = f"Family: {family_code}"
    ws_basic["A1"].font = TITLE_FONT

    ws_basic["A3"] = "MoldCode"
    ws_basic["B3"] = fam.get("moldCode")
    ws_basic["A4"] = "Brands"
    brands = fam.get("brands") or []
    ws_basic["B4"] = ", ".join(map(str, brands)) if isinstance(brands, list) else str(brands)
    ws_basic["A5"] = "Notes"
    ws_basic["B5"] = fam.get("notes")

    for r in range(3, 6):
        ws_basic[f"A{r}"].font = BOLD
        ws_basic[f"A{r}"].alignment = LEFT
        ws_basic[f"B{r}"].alignment = LEFT

    # Components table
    df_components = list_components(fam)
    ws_basic["A7"] = "Components"
    ws_basic["A7"].font = TITLE_FONT

    if df_components.empty:
        ws_basic["A8"] = "(no components found)"
    else:
        write_df(ws_basic, df_components, start_row=8, start_col=1, header=True)
        ws_basic.freeze_panes = "A9"

    # Styles table (below components)
    df_styles = list_styles(fam)
    start_row_styles = 8 + (len(df_components) + 2 if not df_components.empty else 2)
    ws_basic[f"A{start_row_styles}"] = "Styles Using This Family"
    ws_basic[f"A{start_row_styles}"].font = TITLE_FONT

    if df_styles.empty:
        ws_basic[f"A{start_row_styles+1}"] = "(no styles listed)"
    else:
        write_df(ws_basic, df_styles, start_row=start_row_styles+1, start_col=1, header=True)

    autosize_columns(ws_basic)

    # -------------------------
    # 2) Sourcing Rule
    # -------------------------
    ws_src = wb.create_sheet("Sourcing Rule")
    df_src = list_sourcing_rules(fam)

    ws_src["A1"] = "Sourcing Rules (Factory × Component → Vendor)"
    ws_src["A1"].font = TITLE_FONT

    if df_src.empty:
        ws_src["A3"] = "(no sourcing rules yet)"
    else:
        write_df(ws_src, df_src, start_row=3, start_col=1, header=True)
        ws_src.freeze_panes = "A4"

    autosize_columns(ws_src)

    # -------------------------
    # 3) Sizing Rule
    # -------------------------
    ws_size = wb.create_sheet("Sizing Rule")
    df_size = list_sizing_rules(fam)

    ws_size["A1"] = "Sizing Rules (fill ShoeSizes as comma-separated list; leave blank if unknown)"
    ws_size["A1"].font = TITLE_FONT

    if df_size.empty:
        ws_size["A3"] = "(no sizing rules block yet)"
    else:
        write_df(ws_size, df_size, start_row=3, start_col=1, header=True)
        ws_size.freeze_panes = "A4"

    autosize_columns(ws_size)

    # -------------------------
    # 4) Mold Inventory – {component} sheets
    # -------------------------
    components = fam.get("components", {}) or {}
    for sole_type, comp_dict in components.items():
        for comp_code, comp in comp_dict.items():
            sheet_title = safe_sheet_name(f"MoldInv_{comp_code}")
            ws_inv = wb.create_sheet(sheet_title)

            ws_inv["A1"] = f"Mold Inventory - {comp_code}"
            ws_inv["A1"].font = TITLE_FONT
            ws_inv["A2"] = f"SoleType: {sole_type}"
            ws_inv["A3"] = f"Component Name: {comp.get('displayName')}"
            ws_inv["A4"] = f"Compound: {comp.get('compound')}"
            for r in range(2, 5):
                ws_inv[f"A{r}"].font = BOLD

            # Build matrix
            df_matrix, col_meta = build_inventory_matrix_for_component(comp)

            if df_matrix.empty:
                ws_inv["A6"] = "(no mold inventory records)"
                continue

            # Write matrix starting row 6
            write_df(ws_inv, df_matrix, start_row=6, start_col=1, header=True)
            ws_inv.freeze_panes = "B7"

            # Add a metadata block to the right for readability (optional but useful)
            meta_start_col = df_matrix.shape[1] + 3
            ws_inv.cell(row=6, column=meta_start_col, value="Column Details").font = TITLE_FONT

            meta_rows = []
            for m in col_meta:
                meta_rows.append({
                    "ColumnLabel": m["colLabel"],
                    "Vendor": m["vendorName"] or m["vendorCode"],
                    "MoldShop": m["moldShopName"] or m["moldShopCode"],
                    "Location": m["location"],
                    "InitSeason": m["initSeason"],
                    "LastDemand": m["lastDemandSeason"],
                    "TotalMoldQty": m["totalMoldQty"],
                    "DailyOutput": m["dailyOutput"],
                    "MoldCost": m["moldCost"],
                    "Ownership": m["ownership"],
                    "Condition": m["condition"]
                })
            df_meta = pd.DataFrame(meta_rows)
            write_df(ws_inv, df_meta, start_row=7, start_col=meta_start_col, header=True)

            autosize_columns(ws_inv)

    # -------------------------
    # Save
    # -------------------------
    os.makedirs(output_dir, exist_ok=True)
    out_path = os.path.join(output_dir, f"{family_code}.xlsx")
    wb.save(out_path)
    return out_path


def export_all_families(json_path, output_dir):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    families = data.get("families", {})
    if not families:
        raise ValueError("No 'families' found in JSON.")

    written = []
    for family_code, fam in families.items():
        path = export_one_family_to_excel(family_code, fam, output_dir)
        written.append(path)

    return written


# ====== EDIT THESE ======
JSON_PATH = "../data/mold_data.json"          # <-- your master v2 json
OUTPUT_DIR = "../data/output/"     # <-- folder output
# ========================

files = export_all_families(JSON_PATH, OUTPUT_DIR)
print(f"Exported {len(files)} family workbook(s) to: {OUTPUT_DIR}")
for p in files:
    print(" -", p)


FileNotFoundError: [Errno 2] No such file or directory: '../data/output/MRS-846/853.xlsx'